In [1]:
import pandas as pd

In [2]:
inventario_demanda_df = pd.read_csv('../data/_raw/inventario_demanda.csv')
inventario_demanda_df

,fecha,id_producto,nombre_producto,categoria,id_sucursal,nombre_sucursal,proveedor,unidades_vendidas,unidades_recibidas,stock_cierre,precio_unitario,costo_unitario,importe_venta,importe_costo
0,2024-06-06,P025,Papel Higiénico 4-pack,Higiene,S01,Mérida Centro,Grupo Abasto del Sur,45,0.0,56,61.01,30.0,2745.45,1350.0
1,2023-08-17,P007,Jabón Multiusos 500ml,Limpieza,S05,Playa del Carmen,Proveedor Central del Caribe,9,0.0,146,41.74,19.0,375.66,171.0
2,2023-12-14,P018,Café Molido 250g,Abarrotes,S04,Cancún Centro,Comercializadora Peninsular,14,0.0,203,84.89,42.0,1188.46,588.0
3,2023-07-09,P006,Detergente Líquido 1L,Limpieza,S04,Cancún Centro,Logística Maya S.A.,16,0.0,218,72.62,35.0,1161.92,560.0
4,2023-11-25,P010,Esponja Cocina Pack 3,Limpieza,S01,Mérida Centro,Comercializadora Peninsular,13,95.0,278,33.61,15.0,436.93,195.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53582,2023-11-17,P006,Detergente Líquido 1L,Limpieza,S02,Mérida Norte,NaN,8,0.0,142,75.26,35.0,602.08,280.0
53583,2024-06-28,P025,Papel Higiénico 4-pack,Higiene,S03,Cancún,Distribuidora Norte S.A.,23,175.0,175,63.07,30.0,1450.61,690.0
53584,2023-12-04,P021,Queso Manchego 400g,Lácteos,S02,Mérida Norte,Grupo Abasto del Sur,8,188.0,188,92.44,50.0,739.52,400.0
53585,2024-03-02,P001,Agua Mineral 1L,Bebidas,S02,Mérida Norte,Comercializadora Peninsular,23,0.0,131,18.69,8.0,429.87,184.0


In [3]:
def inspect(df, view='all', k=1.5):
    """
    Muestra tablas para inspeccionar el DataFrame.
    
    Parámetros
    ----------
    df : pd.DataFrame
    view : str, default 'all'
        - 'all' -> tabla resumen de columnas + tabla de duplicados
        - nombre de columna -> value_counts completo de esa columna + outliers
    k : float, default 1.5
        Multiplicador del IQR para detectar outliers (1.5 = estándar, 3 = solo extremos)
    """
    import pandas as pd
    from IPython.display import display
    
    if view == 'all':
        desc = df.describe(include="all").T
        desc["n_unique"] = df.nunique()
        desc["null_count"] = df.isnull().sum()
        desc["skewness"] = df.skew(numeric_only=True)
        desc["kurtosis"] = df.kurtosis(numeric_only=True)
        desc["type"] = df.dtypes
        
        num = df.select_dtypes(include="number")
        q1 = num.quantile(0.25)
        q3 = num.quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - k * iqr
        lim_sup = q3 + k * iqr
        es_outlier = (num < lim_inf) | (num > lim_sup)
        
        desc["n_outliers"] = es_outlier.sum()
        
        desc = desc.drop(columns=["unique", "top", "freq"], errors="ignore")
        
        orden = ["type", "count", "null_count", "n_unique",
                 "mean", "std", "min", "25%", "50%", "75%", "max",
                 "n_outliers",
                 "skewness", "kurtosis"]
        desc = desc[[c for c in orden if c in desc.columns]]
        
        print("=" * 60 + " RESUMEN DE COLUMNAS " + "=" * 60)
        display(desc)
        
        dups = df[df.duplicated(keep=False)].copy()
        if not dups.empty:
            dups = dups.sort_values(by=list(df.columns))
            print("\n" + "=" * 60 + " FILAS DUPLICADAS (muestra hasta 50) " + "=" * 60)
            display(dups.head(50))
            if len(dups) > 50:
                print(f"... y {len(dups)-50} filas duplicadas más.")
        else:
            print("\n--- No hay filas duplicadas ---")
        
        return None
    
    else:
        if view not in df.columns:
            print(f"La columna '{view}' no existe en el DataFrame.")
            return None
        
        serie = df[view]
        print(f"\nColumna: {view}")
        print(f"Tipo: {serie.dtype}")
        print(f"Nulos: {serie.isnull().sum()} ({serie.isnull().mean():.2%})")
        print(f"Valores únicos: {serie.nunique()}\n")
        
        if pd.api.types.is_numeric_dtype(serie):
            q1, q3 = serie.quantile([0.25, 0.75])
            iqr = q3 - q1
            lim_inf, lim_sup = q1 - k * iqr, q3 + k * iqr
            mask = (serie < lim_inf) | (serie > lim_sup)
            n_out = mask.sum()
            
            print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
            print(f"Límites (k={k}): {lim_inf:.2f} a {lim_sup:.2f}")
            print(f"Outliers: {n_out} ({n_out / serie.notna().sum():.2%})"
                  f"  -> {(serie < lim_inf).sum()} abajo, {(serie > lim_sup).sum()} arriba\n")
            
            if n_out > 0:
                print("=" * 30 + " FILAS CON OUTLIERS (muestra hasta 50) " + "=" * 30)
                display(df[mask].sort_values(view).head(50))
                print()
        
        vc = serie.value_counts(dropna=False)
        vc_df = vc.reset_index()
        vc_df.columns = [view, 'count']
        vc_df['percentage'] = (100 * vc_df['count'] / len(df)).round(2)
        
        if len(vc_df) > 200:
            print(f"(La columna tiene {len(vc_df)} valores únicos, puede tardar en mostrarse)")
        
        with pd.option_context('display.max_rows', None):
            display(vc_df)
        
        return None

In [9]:
inspect(inventario_demanda_df, view="all")

============================================================ RESUMEN DE COLUMNAS ============================================================


,type,count,null_count,n_unique,mean,std,min,25%,50%,75%,max,n_outliers,skewness,kurtosis
fecha,object,53587,0,970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_producto,object,53587,0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_producto,object,53587,0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,object,53587,0,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_sucursal,object,53587,0,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_sucursal,object,53587,0,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
proveedor,object,51444,2143,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unidades_vendidas,int64,53587.0,0,87,21.461343,16.545298,0.0,9.0,17.0,30.0,86.0,1076.0,1.035678,0.333272
unidades_recibidas,float64,50908.0,2679,272,21.65418,60.459853,0.0,0.0,0.0,0.0,320.0,7239.0,3.034606,8.728954
stock_cierre,int64,53587.0,0,1603,256.541008,255.464259,0.0,115.0,177.0,267.0,1794.0,6536.0,2.595045,7.595903



============================================================ FILAS DUPLICADAS (muestra hasta 50) ============================================================


,fecha,id_producto,nombre_producto,categoria,id_sucursal,nombre_sucursal,proveedor,unidades_vendidas,unidades_recibidas,stock_cierre,precio_unitario,costo_unitario,importe_venta,importe_costo
23690,2023-01-01,P008,Cloro Blanqueador 1L,Limpieza,S04,Cancún Centro,Grupo Abasto del Sur,3,0.0,536,28.46,12.0,85.38,36.0
40590,2023-01-01,P008,Cloro Blanqueador 1L,Limpieza,S04,Cancún Centro,Grupo Abasto del Sur,3,0.0,536,28.46,12.0,85.38,36.0
44436,2023-01-03,P005,Cerveza Lata 355ml,Bebidas,S04,Cancún Centro,Proveedor Central del Caribe,49,0.0,139,28.20,14.0,1381.80,686.0
52727,2023-01-03,P005,Cerveza Lata 355ml,Bebidas,S04,Cancún Centro,Proveedor Central del Caribe,49,0.0,139,28.20,14.0,1381.80,686.0
12828,2023-01-06,P015,Azúcar Blanca 1kg,Abarrotes,S01,Mérida Centro,Distribuidora Norte S.A.,22,0.0,199,28.10,14.0,618.20,308.0
27906,2023-01-06,P015,Azúcar Blanca 1kg,Abarrotes,S01,Mérida Centro,Distribuidora Norte S.A.,22,0.0,199,28.10,14.0,618.20,308.0
9941,2023-01-07,P003,Jugo Naranja 1L,Bebidas,S05,Playa del Carmen,Grupo Abasto del Sur,14,0.0,303,35.38,16.0,495.32,224.0
42295,2023-01-07,P003,Jugo Naranja 1L,Bebidas,S05,Playa del Carmen,Grupo Abasto del Sur,14,0.0,303,35.38,16.0,495.32,224.0
14247,2023-01-09,P017,Sopa Pasta 200g,Abarrotes,S04,Cancún Centro,Grupo Abasto del Sur,35,0.0,176,15.10,7.0,528.50,245.0
25380,2023-01-09,P017,Sopa Pasta 200g,Abarrotes,S04,Cancún Centro,Grupo Abasto del Sur,35,0.0,176,15.10,7.0,528.50,245.0


... y 456 filas duplicadas más.


In [5]:
fechas = pd.to_datetime(inventario_demanda_df["fecha"], format="%Y-%m-%d", errors="coerce")
inventario_demanda_df.loc[fechas.isna(), "fecha"].value_counts(dropna=False).head(20)

fecha
2024-03             14
2023-06             13
2024-05             12
2023-08             12
2023-11             12
2024-04             11
2023-10             11
2023-01             11
2024-06             10
2023-03             10
2024-01              9
2023-09              9
2023-12              8
2024-02              7
2023-02              6
2023-05              6
2023-04              6
2023-07              5
2024-06-26T00:00     3
2023-05-10T00:00     3
Name: count, dtype: int64

In [6]:
clean_inventario_demanda_df = (
    inventario_demanda_df
    .assign(
        fecha=lambda d: pd.to_datetime(d["fecha"].str[:10], format="%Y-%m-%d", errors="coerce"),
        categoria=lambda d: (
            d['categoria']
            .str.strip()
            .str.lower()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("utf-8")
            .replace({
                "abarrote": "abarrotes",
                "bebida": "bebidas",
                "higiene personal": "higiene",
            })
            .map({
                "abarrotes": "Abarrotes",
                "bebidas": "Bebidas",
                "higiene": "Higiene",
                "lacteos": "Lácteos",
                "limpieza": "Limpieza",
            })
            .astype("category")
        ),
        nombre_sucursal=lambda d: (
            d["nombre_sucursal"]
            .str.strip()
            .str.lower()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("utf-8")
            .map({
                "playa del carmen": "Playa del Carmen",
                "cancun": "Cancún",
                "cancun centro": "Cancún Centro",
                "merida norte": "Mérida Norte",
                "merida centro": "Mérida Centro",
            })
            .astype("category")
        ),
        proveedor=lambda d: d["proveedor"].fillna("N/A").astype("category"),
        unidades_recibidas=lambda d: d["unidades_recibidas"].fillna(0).astype("int64"),
    )
    .astype({
        "id_producto": "category",
        "nombre_producto": "category",
        "id_sucursal": "category",
    })
    .dropna(subset=["fecha"])
    .drop_duplicates()
    .reset_index(drop=True)
)

clean_inventario_demanda_df

,fecha,id_producto,nombre_producto,categoria,id_sucursal,nombre_sucursal,proveedor,unidades_vendidas,unidades_recibidas,stock_cierre,precio_unitario,costo_unitario,importe_venta,importe_costo
0,2024-06-06,P025,Papel Higiénico 4-pack,Higiene,S01,Mérida Centro,Grupo Abasto del Sur,45,0,56,61.01,30.0,2745.45,1350.0
1,2023-08-17,P007,Jabón Multiusos 500ml,Limpieza,S05,Playa del Carmen,Proveedor Central del Caribe,9,0,146,41.74,19.0,375.66,171.0
2,2023-12-14,P018,Café Molido 250g,Abarrotes,S04,Cancún Centro,Comercializadora Peninsular,14,0,203,84.89,42.0,1188.46,588.0
3,2023-07-09,P006,Detergente Líquido 1L,Limpieza,S04,Cancún Centro,Logística Maya S.A.,16,0,218,72.62,35.0,1161.92,560.0
4,2023-11-25,P010,Esponja Cocina Pack 3,Limpieza,S01,Mérida Centro,Comercializadora Peninsular,13,95,278,33.61,15.0,436.93,195.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52734,2023-11-17,P006,Detergente Líquido 1L,Limpieza,S02,Mérida Norte,N/A,8,0,142,75.26,35.0,602.08,280.0
52735,2024-06-28,P025,Papel Higiénico 4-pack,Higiene,S03,Cancún,Distribuidora Norte S.A.,23,175,175,63.07,30.0,1450.61,690.0
52736,2023-12-04,P021,Queso Manchego 400g,Lácteos,S02,Mérida Norte,Grupo Abasto del Sur,8,188,188,92.44,50.0,739.52,400.0
52737,2024-03-02,P001,Agua Mineral 1L,Bebidas,S02,Mérida Norte,Comercializadora Peninsular,23,0,131,18.69,8.0,429.87,184.0


In [7]:
inspect(clean_inventario_demanda_df, view="all")

============================================================ RESUMEN DE COLUMNAS ============================================================


,type,count,null_count,n_unique,mean,std,min,25%,50%,75%,max,n_outliers,skewness,kurtosis
fecha,datetime64[ns],52739,0,547,2023-10-01 10:11:30.388517376,NaN,2023-01-01 00:00:00,2023-05-17 00:00:00,2023-10-02 00:00:00,2024-02-15 00:00:00,2024-06-30 00:00:00,NaN,NaN,NaN
id_producto,category,52739,0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_producto,category,52739,0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,category,52739,0,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_sucursal,category,52739,0,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_sucursal,category,52739,0,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
proveedor,category,52739,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unidades_vendidas,int64,52739.0,0,87,21.458484,16.547542,0.0,9.0,17.0,30.0,86.0,1060.0,1.036175,0.333656
unidades_recibidas,int64,52739.0,0,272,20.547792,59.137436,0.0,0.0,0.0,0.0,320.0,7105.0,3.140029,9.438052
stock_cierre,int64,52739.0,0,1598,256.461442,255.245322,0.0,115.0,177.0,267.0,1794.0,6431.0,2.595001,7.600282



--- No hay filas duplicadas ---


In [8]:
clean_inventario_demanda_df.to_parquet('../data/processed/inventario_demanda_limpio.parquet')